# SparkCut v3 - RNN propia para detectar fragmentos con potencial viral

**Entorno de ejecución > GPU (T4).** Tiempo total aproximado: 60-90 min (el etiquetado con LLM es la parte más lenta).

## Qué corrige esta versión (diagnóstico de la v2)
En la v2, el puntaje dependía de la **posición** del fragmento en el video (la misma frase valía 27 puntos más al inicio) y puntuaba más alto el texto neutro que el emocional. La métrica "RNN vs solo-posición" fue casi idéntica: el texto aportaba muy poco. Causas: (1) la etiqueta de repetición de YouTube no mide viralidad emocional y (2) la posición contamina la señal.

## Qué hace la v3
1. **Modelo solo-texto**: la RNN no recibe la posición. El puntaje no puede depender de dónde está el fragmento.
2. **Objetivo des-sesgado**: se aprende "cuánto se repite el fragmento *por encima de lo que predice su posición*" (predictor de posición con validación cruzada por video).
3. **Profesor LLM alineado al producto**: un LLM puntúa miles de ventanas con una rúbrica de potencial viral (gancho, emoción, sorpresa, conflicto, historia). Tu RNN se entrena para reproducir esas etiquetas (**destilación**: la RNN, el tokenizador y el entrenamiento son tuyos; declara el profesor en tu informe). Requiere una clave de API en Colab (Secrets: `ANTHROPIC_API_KEY`); sin clave, el notebook corre solo con el objetivo de repetición y lo avisa.
4. **Ensamble de 3 RNN** (semillas distintas) y regularización fuerte (dropout, *word dropout*, early stopping).
5. **Evaluación estricta**: líneas base, intervalos de confianza por *bootstrap* sobre videos, prueba de independencia de la posición, fidelidad al profesor y **conjunto manual** etiquetado por dos personas (acuerdo entre evaluadores).

> Datos: `behavior-in-the-wild/content-behavior-corpus`, `google-research-datasets/go_emotions` y `dair-ai/emotion` (Hugging Face).

In [ ]:
!pip install -q datasets sentencepiece scikit-learn scipy pandas anthropic

In [ ]:
import os, json, re, zlib, random, shutil, hashlib, time, math
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as Fn
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from datasets import load_dataset
from scipy.stats import spearmanr, rankdata
from sklearn.metrics import roc_auc_score
device = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
def seed_all(s): random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(SEED)
CFG = dict(
    MAX_TRAIN_VIDEOS=22000, MAX_TEST_VIDEOS=3000, AUX_SAMPLES=40000,
    WIN_TARGET=25, WIN_MIN=10, WIN_MAX=45, HOP=8, MIN_WORDS=12, MIN_LEXDIV=0.30, MIN_WIN_PER_VIDEO=5, MIN_VIDEO_LEN=45, PEAK_Q=0.8,
    MAX_LEN=160, VOCAB=12000, EMB=192, HID=160, LAYERS=2, DROP=0.4, WORD_DROP=0.10,
    EPOCHS=25, PATIENCE=4, BS=128, LR=1.5e-3, N_SEEDS=3, TEACHER_REPEAT=3,
    LOSS_W=dict(adj=1.0, pk=0.5, tv=4.0, td=1.0, aux=0.3),
    SCORE_W_TEACHER=dict(tv=0.6, adj=0.2, pk=0.2), SCORE_W_REPLAY=dict(tv=0.0, adj=0.5, pk=0.5),
    N_TEACHER_TRAIN=5000, N_TEACHER_VAL=600, N_TEACHER_TEST=600, N_MANUAL=250,
    LLM_MODEL='claude-haiku-5-5', LLM_BATCH=8, LLM_WORKERS=6, USE_DRIVE=True)
print('Dispositivo:', device)

Dispositivo: cuda


## 1. Content Behavior Corpus: ventanas de habla con su etiqueta de repeticion

In [ ]:
cbc = load_dataset('behavior-in-the-wild/content-behavior-corpus')
fin = lambda v: v is not None and bool(np.isfinite(float(v)))      # descarta None / NaN / inf
WHY = Counter()

def norm_video(ex):
    asr = [s for s in (ex.get('asr_raw') or []) if s and s.get('text') and str(s['text']).strip() and fin(s.get('start')) and fin(s.get('end'))]
    if len(asr) < 3: WHY['pocos_segmentos_asr'] += 1; return None
    mk = [m for m in ((ex.get('replay_graphs') or {}).get('markers') or []) if m and fin(m.get('replay_score')) and fin(m.get('start')) and fin(m.get('end'))]
    if not mk: WHY['sin_marcadores_replay'] += 1; return None
    amax = max(float(s['end']) for s in asr)
    try: length = float((ex.get('metadata') or {}).get('length') or 0)
    except Exception: length = 0.0
    if not np.isfinite(length) or length <= 0: length = amax          # (v2 dejaba pasar length=NaN: causa de los 'pos' no finitos)
    a_scale = 0.001 if amax > length * 5 else 1.0
    segs = [dict(start=float(s['start']) * a_scale, end=float(s['end']) * a_scale, text=str(s['text']).strip()) for s in asr]
    ms = np.array([float(m['start']) for m in mk]); me = np.array([float(m['end']) for m in mk]); mv = np.array([float(m['replay_score']) for m in mk])
    f = length / me.max() if me.max() > 0 else 1.0
    if abs(f - 1) > 0.05: ms, me = ms * f, me * f
    if length < CFG['MIN_VIDEO_LEN']: WHY['video_muy_corto'] += 1; return None
    return segs, ms, me, mv, length

def make_windows(segs, ms, me, mv):
    out, i = [], 0
    while i < len(segs):
        j = i
        while j + 1 < len(segs) and segs[j]['end'] - segs[i]['start'] < CFG['WIN_TARGET'] and segs[j + 1]['end'] - segs[i]['start'] <= CFG['WIN_MAX']: j += 1
        st, en = segs[i]['start'], segs[j]['end']; text = ' '.join(s['text'] for s in segs[i:j + 1]); words = text.lower().split()
        if en - st >= CFG['WIN_MIN'] and len(words) >= CFG['MIN_WORDS'] and len(set(words)) / len(words) >= CFG['MIN_LEXDIV']:   # limpia "Ah! Ah! Ah!..."
            ov = np.clip(np.minimum(en, me) - np.maximum(st, ms), 0, None)
            if ov.sum() > 0:
                raw = float((ov * mv).sum() / ov.sum())
                if np.isfinite(raw): out.append(dict(text=text, start=st, end=en, raw=raw))
        k = i + 1
        while k < len(segs) and segs[k]['start'] < st + CFG['HOP']: k += 1
        i = k
    return out

def pos_feats(start, end, L): return [start / L, (end - start) / L, float(start / L < 0.05), math.log(max(L, 1.0)) / 10]

def video_rows(ex):
    nv = norm_video(ex)
    if nv is None: return []
    segs, ms, me, mv, L = nv; w = make_windows(segs, ms, me, mv)
    if len(w) < CFG['MIN_WIN_PER_VIDEO']: WHY['pocas_ventanas'] += 1; return []
    WHY['video_ok'] += 1
    pct = (rankdata([x['raw'] for x in w]) - 1) / (len(w) - 1)         # percentil de repeticion dentro del propio video
    return [dict(text=x['text'], vid=str(ex['video_id']), start=x['start'], end=x['end'], L=L, pct=float(p), pkraw=float(p >= CFG['PEAK_Q']),
                 pos=pos_feats(x['start'], x['end'], L)) for x, p in zip(w, pct)]

SEEN = set()
def build_cbc(split, n):
    ds = cbc[split].shuffle(seed=SEED); ds = ds.select(range(min(n, len(ds)))); rows = []
    for k, ex in enumerate(ds):
        for r in video_rows(ex):
            h = hashlib.md5(r['text'].encode()).hexdigest()
            if h in SEEN: WHY['duplicada'] += 1; continue            # sin duplicados (tambien entre train y test)
            SEEN.add(h); rows.append(r)
        if k % 2000 == 0: print(split, k, 'videos ->', len(rows), 'ventanas')
    return rows

main_all = build_cbc('train', CFG['MAX_TRAIN_VIDEOS']); main_test = build_cbc('test', CFG['MAX_TEST_VIDEOS'])
print('Motivos de descarte:', dict(WHY))
is_val = lambda r: zlib.crc32(r['vid'].encode()) % 10 == 0           # validacion por VIDEO (sin fuga)
main_val = [r for r in main_all if is_val(r)]; main_train = [r for r in main_all if not is_val(r)]
print(len(main_train), len(main_val), len(main_test), 'ventanas (train/val/test) |', len({r['vid'] for r in main_train}), 'videos de train')
assert len(main_train) > 500, 'Muy pocas ventanas'
for r in random.sample(main_train, 3): print(round(r['pct'], 2), '|', r['text'][:130])

README.md:   0%|          | 0.00/9.44k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  124MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 11.1MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/22569 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2026 [00:00<?, ? examples/s]

train 0 videos -> 0 ventanas
train 2000 videos -> 3066 ventanas
train 4000 videos -> 6035 ventanas
train 6000 videos -> 9344 ventanas
train 8000 videos -> 12717 ventanas
train 10000 videos -> 16041 ventanas
train 12000 videos -> 19411 ventanas
train 14000 videos -> 22709 ventanas
train 16000 videos -> 25982 ventanas
train 18000 videos -> 29504 ventanas
train 20000 videos -> 32982 ventanas
test 0 videos -> 0 ventanas
test 2000 videos -> 3386 ventanas
Motivos de descarte: {'pocas_ventanas': 7206, 'video_ok': 6217, 'pocos_segmentos_asr': 7303, 'video_muy_corto': 3300, 'duplicada': 51}
32864 3407 3437 ventanas (train/val/test) | 5130 videos de train
0.2 | called Queen's View. It just makes you feel so good to see the beautiful, beautiful water. It's just so relaxing. At the end of th
1.0 | I've done kickboxing. I've done Muay Thai. Although both are stand-up fighting styles, they are quite different. In kickboxing, yo
0.5 | Sloppy play! You're out. That is so fucking weak, Doug. I gave you

## 2. Objetivo des-sesgado: repeticion por encima de lo que predice la posicion

In [ ]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupKFold
Pm = lambda rows: np.array([r['pos'] for r in rows]); Y = lambda rows, k: np.array([r[k] for r in rows])
mk_hgb = lambda: HistGradientBoostingRegressor(max_iter=150, learning_rate=0.08, random_state=SEED)
Xtr, ytr, gtr = Pm(main_train), Y(main_train, 'pct'), np.array([r['vid'] for r in main_train])
oof = np.zeros(len(ytr))
for a, b in GroupKFold(5).split(Xtr, ytr, gtr): oof[b] = mk_hgb().fit(Xtr[a], ytr[a]).predict(Xtr[b])   # prediccion "fuera de bolsa" por video
pos_model = mk_hgb().fit(Xtr, ytr)
res_tr = ytr - oof; srt = np.sort(res_tr)
to_adj = lambda res: np.searchsorted(srt, res, side='right') / len(srt)            # percentil global del residuo
for rows, pp in ((main_train, oof), (main_val, pos_model.predict(Pm(main_val))), (main_test, pos_model.predict(Pm(main_test)))):
    for r, p in zip(rows, pp):
        r['ppos'] = float(p); r['adj'] = float(to_adj(r['pct'] - p)); r['pk'] = float(r['adj'] >= CFG['PEAK_Q'])
print('R2 del predictor de posicion en validacion: %.3f' % (1 - np.var(Y(main_val, 'pct') - Y(main_val, 'ppos')) / np.var(Y(main_val, 'pct'))))
print('Corr(adj, posicion relativa) en train: %.3f  <- debe ser ~0' % spearmanr(Y(main_train, 'adj'), [r['pos'][0] for r in main_train])[0])

R2 del predictor de posicion en validacion: 0.030
Corr(adj, posicion relativa) en train: -0.021  <- debe ser ~0


## 3. Conjunto manual para tus evaluadores (Kevin y Andres) + seleccion de ventanas para el profesor

In [ ]:
rng = random.Random(SEED)
def pick(rows, n, per_video=2, taken=None):
    taken = taken if taken is not None else set(); by = {}
    for i, r in enumerate(rows): by.setdefault(r['vid'], []).append(i)
    vids = list(by); rng.shuffle(vids); out = []
    for _ in range(per_video):
        for v in vids:
            if len(out) >= n: return [rows[i] for i in out]
            cand = [i for i in by[v] if i not in taken]
            if cand: i = rng.choice(cand); taken.add(i); out.append(i)
    return [rows[i] for i in out]
tk = set(); manual_rows = pick(main_test, CFG['N_MANUAL'], 1, tk)                                   # 1 ventana por video de test
teach_test = manual_rows + pick(main_test, CFG['N_TEACHER_TEST'] - len(manual_rows), 2, tk)
teach_val = pick(main_val, CFG['N_TEACHER_VAL'], 2); teach_train = pick(main_train, CFG['N_TEACHER_TRAIN'], 3)
manual_df = pd.DataFrame(dict(id=range(len(manual_rows)), text=[r['text'] for r in manual_rows], kevin_score='', andres_score=''))
manual_df.to_csv('manual_template.csv', index=False)
print(len(manual_rows), 'fragmentos para etiquetar a mano -> manual_template.csv')
print('INSTRUCCION: Kevin y Andres puntuan CADA UNO, por separado (sin hablar), de 0 a 100 el potencial de ese fragmento como short viral.')
print('Rubrica: 0-20 relleno/introducciones, 20-40 informativo pero plano, 40-60 interesante, 60-80 fuerte, 80-100 excepcional. Llenen kevin_score y andres_score y guarden como manual_labeled.csv.')
try:
    from google.colab import files; files.download('manual_template.csv')
except Exception as e: print('(descarga manual: archivo manual_template.csv en el panel de archivos)')

250 fragmentos para etiquetar a mano -> manual_template.csv
INSTRUCCION: Kevin y Andres puntuan CADA UNO, por separado (sin hablar), de 0 a 100 el potencial de ese fragmento como short viral.
Rubrica: 0-20 relleno/introducciones, 20-40 informativo pero plano, 40-60 interesante, 60-80 fuerte, 80-100 excepcional. Llenen kevin_score y andres_score y guarden como manual_labeled.csv.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 4. Profesor LLM: puntaje de potencial viral con rubrica (etiquetas para destilar)

In [ ]:
import os
KEY = None
try:
    from google.colab import userdata; KEY = userdata.get('ANTHROPIC_API_KEY')
except Exception: KEY = os.environ.get('ANTHROPIC_API_KEY')
USE_TEACHER = bool(KEY)
if not USE_TEACHER:
    print('AVISO: no hay ANTHROPIC_API_KEY (Colab > Secrets). El modelo se entrenara SOLO con repeticion + emociones (calidad menor).')
CACHE_PATH = 'teacher_cache.json'
if CFG['USE_DRIVE'] and USE_TEACHER:
    try:
        from google.colab import drive; drive.mount('/content/drive'); os.makedirs('/content/drive/MyDrive/sparkcut', exist_ok=True)
        CACHE_PATH = '/content/drive/MyDrive/sparkcut/teacher_cache.json'
    except Exception as e: print('Drive no disponible; cache local:', type(e).__name__)
DIMS = ['hook', 'emotion', 'surprise', 'conflict', 'story']
SYSTEM = """You are an expert editor of short-form social video (TikTok, Reels, YouTube Shorts). You will receive transcript excerpts (10-45 seconds of speech), each tagged [i].
For EACH excerpt judge how well it would work as a standalone viral short. Give integer scores:
- hook (0-10): grabs attention immediately.
- emotion (0-10): emotional intensity (joy, anger, fear, awe, laughter, tension).
- surprise (0-10): unexpected reveal, twist, bold or counter-intuitive claim.
- conflict (0-10): controversy, strong opinion, humor, tension, stakes.
- story (0-10): self-contained, coherent, satisfying without extra context.
- overall (0-100): holistic viral potential. Anchors: 0-20 filler, greetings, logistics; 20-40 informative but flat; 40-60 interesting; 60-80 strong, shareable; 80-100 exceptional. Most ordinary speech must score below 40.
Reply with ONLY a JSON list, one object per excerpt: [{"id": 0, "hook": 0, "emotion": 0, "surprise": 0, "conflict": 0, "story": 0, "overall": 0}]"""
cache = json.load(open(CACHE_PATH)) if os.path.exists(CACHE_PATH) else {}
ckey = lambda t: hashlib.sha1(t.encode()).hexdigest()

def parse_labels(txt, n):
    a, b = txt.find('['), txt.rfind(']'); out = {}
    for d in json.loads(txt[a:b + 1]):
        i = int(d['id']); v = [float(d[k]) for k in DIMS] + [float(d['overall'])]
        if 0 <= i < n and all(np.isfinite(v)): out[i] = [min(max(x, 0), 10) / 10 for x in v[:5]] + [min(max(v[5], 0), 100) / 100]
    return out
if USE_TEACHER:
    import anthropic; client = anthropic.Anthropic(api_key=KEY)
def label_batch(texts, temperature=0.0):
    body = '\n\n'.join(f'[{i}]\n{t[:1200]}' for i, t in enumerate(texts)); kw = dict(temperature=temperature); last = None
    for attempt in range(6):
        try:
            r = client.messages.create(model=CFG['LLM_MODEL'], max_tokens=1500, system=SYSTEM, messages=[{'role': 'user', 'content': body}], **kw)
            out = parse_labels(r.content[0].text, len(texts))
            if len(out) == len(texts): return out
        except Exception as e:
            last = e
            if 'temperature' in str(e).lower(): kw = {}
        time.sleep(min(2 ** attempt, 30))
    raise RuntimeError(f'Fallo el etiquetado LLM: {last}')

def label_rows(rows, name):
    need = [r for r in rows if ckey(r['text']) not in cache]
    est_in = sum(len(r['text']) / 4 + 30 for r in need) + 330 * (len(need) // CFG['LLM_BATCH'] + 1)
    print(f'{name}: {len(rows)} ventanas, {len(need)} por etiquetar (~{est_in / 1e6:.2f}M tokens de entrada)')
    batches = [need[i:i + CFG['LLM_BATCH']] for i in range(0, len(need), CFG['LLM_BATCH'])]
    def run(bt): return bt, label_batch([r['text'] for r in bt])
    with ThreadPoolExecutor(CFG['LLM_WORKERS']) as ex:
        for k, (bt, out) in enumerate(ex.map(run, batches)):
            for i, r in enumerate(bt): cache[ckey(r['text'])] = out[i]
            if k % 25 == 0: json.dump(cache, open(CACHE_PATH, 'w')); print(f'  {name}: lote {k + 1}/{len(batches)}')
    json.dump(cache, open(CACHE_PATH, 'w'))
    for r in rows: r['teach'] = cache.get(ckey(r['text']))
for r in main_all + main_test: r['teach'] = None
if USE_TEACHER:
    label_batch([teach_val[0]['text'], teach_val[1]['text']])
    for rows, nm in ((teach_val, 'val'), (teach_test, 'test'), (teach_train, 'train')): label_rows(rows, nm)
    tv_all = np.array([r['teach'][5] for r in teach_train + teach_val + teach_test])
    print('Profesor: media %.1f | mediana %.1f | p90 %.1f | %% >=70: %.1f' % (tv_all.mean() * 100, np.median(tv_all) * 100, np.percentile(tv_all, 90) * 100, (tv_all >= .7).mean() * 100))
    srt_ = sorted(teach_train, key=lambda r: r['teach'][5])
    print("\nMENOR potencial:", srt_[0]['text'][:140], "\nMAYOR potencial:", srt_[-1]['text'][:140])

AVISO: no hay ANTHROPIC_API_KEY (Colab > Secrets). El modelo se entrenara SOLO con repeticion + emociones (calidad menor).


NameError: name 'CFG' is not defined

In [ ]:
RESULTS = dict(config={k: v for k, v in CFG.items()}, uses_teacher=USE_TEACHER)
if USE_TEACHER:
    # Consistencia del profesor: se re-etiquetan 120 ventanas con OTRA composicion de lotes y se compara
    sub = rng.sample(teach_val, min(120, len(teach_val))); rng.shuffle(sub); again = {}
    for i in range(0, len(sub), 5):
        out = label_batch([r['text'] for r in sub[i:i + 5]]);
        for j, r in enumerate(sub[i:i + 5]): again[ckey(r['text'])] = out[j][5]
    a = [r['teach'][5] for r in sub]; b = [again[ckey(r['text'])] for r in sub]
    RESULTS['teacher_self_consistency_spearman'] = float(spearmanr(a, b)[0])
    print('Consistencia del profesor (Spearman entre dos pasadas): %.3f' % RESULTS['teacher_self_consistency_spearman'])

## 5. Datos auxiliares de emociones (explican el puntaje)

In [ ]:
GO_IDS = ['google-research-datasets/go_emotions', 'go_emotions']; EM_IDS = ['dair-ai/emotion', 'emotion']
def load_first(ids, *a):
    for i in ids:
        try: print('Cargando', i); return load_dataset(i, *a)
        except Exception as e: print('  fallo:', type(e).__name__)
    raise RuntimeError(f'No se pudo cargar: {ids}')
go = load_first(GO_IDS, 'simplified'); em = load_first(EM_IDS)
GO_NAMES = go['train'].features['labels'].feature.names; EM_NAMES = em['train'].features['label'].names; NGO, NEM = len(GO_NAMES), len(EM_NAMES)
aux_all = []
for r in go['train']:
    y = [0.0] * NGO
    for l in r['labels']: y[l] = 1.0
    aux_all.append(dict(text=r['text'], go=y, gm=1.0, em=-100))
for r in em['train']: aux_all.append(dict(text=r['text'], go=None, gm=0.0, em=r['label']))
random.shuffle(aux_all); aux_val = aux_all[:3000]; aux_train = aux_all[3000:3000 + CFG['AUX_SAMPLES']]
print(len(aux_train), 'filas auxiliares de entrenamiento')

Cargando google-research-datasets/go_emotions


README.md:   0%|          | 0.00/9.40k [00:00<?, ?B/s]

simplified/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.77MB            

simplified/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

simplified/validation-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B /  350kB            

simplified/validation-00000-of-00001.par(…): downloading bytes:           |  0.00B            

... (13 outputs left out)

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

40000 filas auxiliares de entrenamiento


## 6. Tokenizador propio (SentencePiece desde cero) y datos de entrenamiento

In [ ]:
import sentencepiece as spm
ZERO28, ZERO5 = [0.0] * NGO, [0.0] * 5
def mkrow(text, adj=0., pk=0., wr=0., teach=None, go=None, gm=0., em=-100):
    return dict(text=text, adj=adj, pk=pk, wr=wr, tv=(teach[5] if teach else 0.), td=(list(teach[:5]) if teach else ZERO5),
                wt=1.0 if teach else 0.0, go=go or ZERO28, gm=gm, em=em)
cbc_row = lambda r: mkrow(r['text'], r['adj'], r['pk'], 1.0, r.get('teach'))
aux_row = lambda r: mkrow(r['text'], go=r['go'], gm=r['gm'], em=r['em'])
tr_cbc = [cbc_row(r) for r in main_train]; va_cbc = [cbc_row(r) for r in main_val]
train_rows = tr_cbc + [x for x in tr_cbc if x['wt'] > 0] * (CFG['TEACHER_REPEAT'] - 1) + [aux_row(r) for r in aux_train]; random.shuffle(train_rows)
val_rows = va_cbc + [aux_row(r) for r in aux_val]
print(len(train_rows), 'filas de train |', sum(x['wt'] > 0 for x in train_rows), 'con etiqueta del profesor |', len(val_rows), 'de validacion')
open('corpus.txt', 'w').write('\n'.join(r['text'].replace('\n', ' ') for r in random.sample(train_rows, min(150000, len(train_rows)))))
spm.SentencePieceTrainer.train(input='corpus.txt', model_prefix='tokenizer', vocab_size=CFG['VOCAB'], model_type='unigram', pad_id=0, unk_id=1,
    bos_id=-1, eos_id=-1, character_coverage=1.0, input_sentence_size=150000, shuffle_input_sentence=True, hard_vocab_limit=False)
sp = spm.SentencePieceProcessor(model_file='tokenizer.model'); VOCAB = sp.get_piece_size(); MAXL = CFG['MAX_LEN']; print('Vocab:', VOCAB)

from torch.utils.data import Dataset, DataLoader
class DS(Dataset):
    def __init__(s, rows): s.rows = rows
    def __len__(s): return len(s.rows)
    def __getitem__(s, i): return s.rows[i]
def make_collate(train):
    def collate(batch):
        ids = []
        for b in batch:
            t = sp.encode(b['text'])[:MAXL] or [1]
            if train and len(t) > 3: t = [x for x in t if random.random() > CFG['WORD_DROP']] or t[:1]     # word dropout
            ids.append(t)
        L = torch.tensor([len(x) for x in ids]); X = torch.zeros(len(ids), int(L.max()), dtype=torch.long)
        for i, x in enumerate(ids): X[i, :len(x)] = torch.tensor(x)
        f = lambda k: torch.tensor([b[k] for b in batch], dtype=torch.float)
        return dict(x=X, l=L, adj=f('adj'), pk=f('pk'), wr=f('wr'), tv=f('tv'), td=f('td'), wt=f('wt'), go=f('go'), gm=f('gm'),
                    em=torch.tensor([b['em'] for b in batch], dtype=torch.long))
    return collate
collate_train, collate_eval = make_collate(True), make_collate(False)
dl_va = DataLoader(DS(val_rows), 256, collate_fn=collate_eval)

72864 filas de train | 0 con etiqueta del profesor | 6407 de validacion
Vocab: 12000


## 7. Modelo: BiGRU + atencion, solo texto (sin posicion)

In [ ]:
HEADS = ['adj', 'pk', 'tv'] + DIMS                                  # 8 salidas principales
class SparkRNN(nn.Module):
    def __init__(s, vocab, emb, hid, layers, drop):
        super().__init__()
        s.emb = nn.Embedding(vocab, emb, padding_idx=0); s.drop = nn.Dropout(drop)
        s.rnn = nn.GRU(emb, hid, layers, batch_first=True, bidirectional=True, dropout=drop)
        s.att = nn.Linear(hid * 2, 1)
        s.h_go = nn.Linear(hid * 2, NGO); s.h_em = nn.Linear(hid * 2, NEM)
        s.h_main = nn.Sequential(nn.Linear(hid * 2, 128), nn.ReLU(), nn.Dropout(drop), nn.Linear(128, len(HEADS)))
    def forward(s, x, lens):
        e = s.drop(s.emb(x))
        p = nn.utils.rnn.pack_padded_sequence(e, lens.cpu(), batch_first=True, enforce_sorted=False)
        o, _ = s.rnn(p); o, _ = nn.utils.rnn.pad_packed_sequence(o, batch_first=True, total_length=x.size(1))
        a = s.att(o).squeeze(-1).masked_fill(x == 0, -1e9); w = torch.softmax(a, 1).unsqueeze(-1)
        h = s.drop((o * w).sum(1))
        return s.h_go(h), s.h_em(h), s.h_main(h), w.squeeze(-1)
new_model = lambda: SparkRNN(VOCAB, CFG['EMB'], CFG['HID'], CFG['LAYERS'], CFG['DROP']).to(device)
print(sum(p.numel() for p in new_model().parameters()) / 1e6, 'M parametros por modelo')

3.159915 M parametros por modelo


## 8. Entrenamiento del ensamble (early stopping por validacion)

In [ ]:
bce = nn.BCEWithLogitsLoss(reduction='none'); ce = nn.CrossEntropyLoss(ignore_index=-100)
to_dev = lambda b: {k: v.to(device) for k, v in b.items()}
def wmean(v, w): return (v * w).sum() / w.sum().clamp(min=1)
def loss_fn(out, b):
    go, em_, mn, _ = out; s = torch.sigmoid(mn); W = CFG['LOSS_W']
    l_adj = wmean((s[:, 0] - b['adj']) ** 2, b['wr']); l_pk = wmean(bce(mn[:, 1], b['pk']), b['wr'])
    l_tv = wmean(Fn.smooth_l1_loss(s[:, 2], b['tv'], reduction='none', beta=0.1), b['wt'])
    l_td = wmean(Fn.smooth_l1_loss(s[:, 3:], b['td'], reduction='none', beta=0.1).mean(1), b['wt'])
    l_go = wmean(bce(go, b['go']).mean(1), b['gm']); l_em = ce(em_, b['em']) if (b['em'] != -100).any() else 0.0
    return W['adj'] * l_adj + W['pk'] * l_pk + W['tv'] * l_tv + W['td'] * l_td + W['aux'] * (l_go + l_em)
@torch.no_grad()
def val_loss(m):
    m.eval(); tot = n = 0
    for b in dl_va:
        b = to_dev(b); l = loss_fn(m(b['x'], b['l']), b)
        if torch.isfinite(l): tot += l.item() * len(b['x']); n += len(b['x'])
    return tot / n if n else float('inf')
def train_one(k):
    seed_all(SEED + k); m = new_model(); path = f'model_{k}.pt'
    dl = DataLoader(DS(train_rows), CFG['BS'], shuffle=True, collate_fn=collate_train)
    opt = torch.optim.AdamW(m.parameters(), lr=CFG['LR'], weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG['LR'], total_steps=CFG['EPOCHS'] * len(dl))
    best, bad, hist = float('inf'), 0, []; torch.save(m.state_dict(), path)
    for ep in range(CFG['EPOCHS']):
        m.train(); tl = ok = skipped = 0
        for b in dl:
            b = to_dev(b); opt.zero_grad(); loss = loss_fn(m(b['x'], b['l']), b)
            if not torch.isfinite(loss): skipped += 1; continue
            loss.backward(); nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step(); sched.step(); tl += loss.item(); ok += 1
        vl = val_loss(m); hist.append((tl / max(ok, 1), vl)); print(f'[modelo {k}] epoca {ep + 1}: train {tl / max(ok, 1):.4f} | val {vl:.4f} | lotes saltados {skipped}')
        if vl < best: best, bad = vl, 0; torch.save(m.state_dict(), path)
        else:
            bad += 1
            if bad >= CFG['PATIENCE']: print('  early stopping'); break
    assert best < float('inf'), 'La validacion nunca fue finita'
    m.load_state_dict(torch.load(path, map_location=device)); m.eval(); return m, best
MODELS, BEST = [], []
for k in range(CFG['N_SEEDS']):
    m, b = train_one(k); MODELS.append(m); BEST.append(b)
print('Mejor val de cada modelo:', [round(x, 4) for x in BEST])

[modelo 0] epoca 1: train 0.9371 | val 0.4366 | lotes saltados 0
[modelo 0] epoca 2: train 0.8513 | val 0.4225 | lotes saltados 0
[modelo 0] epoca 3: train 0.7837 | val 0.3241 | lotes saltados 0
[modelo 0] epoca 4: train 0.6552 | val 0.2730 | lotes saltados 0
[modelo 0] epoca 5: train 0.5900 | val 0.2538 | lotes saltados 0
[modelo 0] epoca 6: train 0.5444 | val 0.2412 | lotes saltados 0
[modelo 0] epoca 7: train 0.5226 | val 0.2346 | lotes saltados 0


## 9. Prediccion del ensamble, puntaje 10-100 y calibracion

In [ ]:
@torch.no_grad()
def forward_all(m, texts):
    dl = DataLoader(DS([mkrow(t) for t in texts]), 256, shuffle=False, collate_fn=collate_eval); m.eval(); out = []
    for b in dl:
        b = to_dev(b); go, em_, mn, _ = m(b['x'], b['l'])
        out.append(torch.cat([torch.sigmoid(mn), torch.sigmoid(go), torch.softmax(em_, 1)], 1).cpu())
    return torch.cat(out).numpy()
def unpack(P): return dict(adj=P[:, 0], pk=P[:, 1], tv=P[:, 2], td=P[:, 3:8], go=P[:, 8:8 + NGO], em=P[:, 8 + NGO:])
def predict_texts(texts, models=None):
    ms = models or MODELS; return unpack(np.mean([forward_all(m, texts) for m in ms], axis=0))
SW = CFG['SCORE_W_TEACHER'] if USE_TEACHER else CFG['SCORE_W_REPLAY']
raw_score = lambda p: SW['tv'] * p['tv'] + SW['adj'] * p['adj'] + SW['pk'] * p['pk']
# calibracion: con profesor -> ajuste lineal raw -> puntaje del profesor (0-100) en validacion; sin profesor -> cuantiles de validacion
if USE_TEACHER:
    tv_txt = [r['text'] for r in teach_val]; x = raw_score(predict_texts(tv_txt)); y = np.array([r['teach'][5] * 100 for r in teach_val])
    a, b0 = np.polyfit(x, y, 1)
    if a <= 0: raise RuntimeError('Calibracion invalida (pendiente <= 0): el modelo no aprendio la senal del profesor')
    CAL = dict(method='linear', a=float(a), b=float(b0))
else:
    qs = np.percentile(raw_score(predict_texts([r['text'] for r in main_val])), np.linspace(0, 100, 101))
    CAL = dict(method='quantile', x=qs.tolist(), y=(10 + 90 * np.linspace(0, 1, 101)).tolist())
def to_score(raw):
    return np.clip(CAL['a'] * raw + CAL['b'], 10, 100) if CAL['method'] == 'linear' else np.interp(raw, CAL['x'], CAL['y'])
@torch.no_grad()
def score_segments(texts):
    # Entrada: lista de textos (en ingles). Salida: puntajes 10-100. La posicion en el video NO influye.
    return np.round(to_score(raw_score(predict_texts(texts))), 1).tolist()
def explain(text):
    p = predict_texts([text]); i = np.argsort(-p['go'][0])[:3]
    return dict(score=float(np.round(to_score(raw_score(p))[0], 1)), dims={d: float(np.round(p['td'][0][j] * 10, 1)) for j, d in enumerate(DIMS)},
                top_emotions=[(GO_NAMES[k], float(np.round(p['go'][0][k], 2))) for k in i])
print('Calibracion:', {k: (round(v, 3) if isinstance(v, float) else v) for k, v in CAL.items() if k in ('method', 'a', 'b')})

## 10. Evaluacion estricta con intervalos de confianza (bootstrap sobre videos)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
test_texts = [r['text'] for r in main_test]
Pt = predict_texts(test_texts); S_test = to_score(raw_score(Pt))
S_each = [to_score(raw_score(unpack(forward_all(m, test_texts)))) for m in MODELS]
tf = TfidfVectorizer(max_features=60000, ngram_range=(1, 2), sublinear_tf=True, min_df=3)
Xtf = tf.fit_transform([r['text'] for r in main_train]); Xte = tf.transform(test_texts)
pred_tf = Ridge(alpha=3.0).fit(Xtf, Y(main_train, 'adj')).predict(Xte)
rnd = np.random.default_rng(SEED).random(len(main_test))

def vid_table(pred, rows, ykey, pkey, k=3):
    by = {}
    for i, r in enumerate(rows): by.setdefault(r['vid'], []).append(i)
    tab = []
    for v, idx in by.items():
        if len(idx) < 4: continue
        p = pred[idx]; y = np.array([rows[i][ykey] for i in idx]); pk = np.array([rows[i][pkey] for i in idx])
        sp_ = spearmanr(p, y)[0] if np.ptp(p) > 0 and np.ptp(y) > 0 else np.nan
        au = roc_auc_score(pk, p) if 0 < pk.sum() < len(pk) and np.ptp(p) > 0 else np.nan
        tab.append([sp_, pk[np.argsort(-p, kind='stable')[:k]].mean(), au])
    return np.array(tab)
def boot(tab, other=None, B=2000):
    g = np.random.default_rng(0); n = len(tab); res = []
    for _ in range(B):
        i = g.integers(0, n, n); a = np.nanmean(tab[i], 0); res.append(a if other is None else a - np.nanmean(other[i], 0))
    est = np.nanmean(tab, 0) if other is None else np.nanmean(tab, 0) - np.nanmean(other, 0)
    return est, np.percentile(np.array(res), [2.5, 97.5], axis=0)
fmt = lambda e, ci, j: f'{e[j]:.3f} [{ci[0][j]:.3f}, {ci[1][j]:.3f}]'
def report(title, methods, ykey, pkey):
    print('\n' + title); tabs = {n: vid_table(p, main_test, ykey, pkey) for n, p in methods.items()}; tab_rows = {}
    for n, t in tabs.items():
        e, ci = boot(t); tab_rows[n] = {'Spearman por video': fmt(e, ci, 0), 'precision@3': fmt(e, ci, 1), 'AUC pico': fmt(e, ci, 2)}
    print(pd.DataFrame(tab_rows).T.to_string()); return tabs, tab_rows
# (A) objetivo des-sesgado: la posicion NO ayuda, solo cuenta el texto
tabsA, rowsA = report('(A) Repeticion des-sesgada (solo texto cuenta). Azar de precision@3 ~ 0.20:',
    {'azar': rnd, 'TF-IDF + Ridge': pred_tf, 'RNN SparkCut (ensamble)': S_test}, 'adj', 'pk')
for ref in ('TF-IDF + Ridge', 'azar'):
    e, ci = boot(tabsA['RNN SparkCut (ensamble)'], tabsA[ref]); print(f'  RNN - {ref}: dSpearman {fmt(e, ci, 0)} | dP@3 {fmt(e, ci, 1)}  (IC95%% que no incluye 0 = diferencia real)')
RESULTS['A_replay_debiased'] = rowsA
# (B) comparable con la v2: repeticion cruda (incluye el efecto de la posicion)
ppos_te = Y(main_test, 'ppos')
tabsB, rowsB = report('(B) Repeticion cruda (comparable con v2; la posicion SI ayuda aqui):',
    {'azar': rnd, 'solo posicion (HGB)': ppos_te, 'RNN SparkCut (solo texto)': S_test}, 'pct', 'pkraw'); RESULTS['B_replay_raw'] = rowsB
# (C) estabilidad entre semillas
RESULTS['C_seeds_P3'] = [float(np.nanmean(vid_table(s, main_test, 'adj', 'pk')[:, 1])) for s in S_each]
print('\n(C) precision@3 por modelo individual:', [round(x, 3) for x in RESULTS['C_seeds_P3']], '| ensamble:', round(float(np.nanmean(tabsA['RNN SparkCut (ensamble)'][:, 1])), 3))
# (D) independencia de la posicion
rho = spearmanr(S_test, [r['pos'][0] for r in main_test])[0]; RESULTS['D_corr_score_vs_position'] = float(rho)
print(f'(D) Correlacion entre puntaje y posicion en el video: {rho:.3f}  (v2: la posicion movia el puntaje 27 puntos; ahora debe ser ~0)')
print('    Distribucion del puntaje en test: media %.1f | %% >70: %.1f' % (S_test.mean(), (S_test > 70).mean() * 100))

In [ ]:
# (E) fidelidad al profesor LLM (ventanas de test que el modelo NUNCA vio) y comparacion con TF-IDF entrenado con las mismas etiquetas
if USE_TEACHER:
    tt = [r['text'] for r in teach_test]; yt = np.array([r['teach'][5] * 100 for r in teach_test]); st = to_score(raw_score(predict_texts(tt)))
    ridge_t = Ridge(alpha=1.0).fit(tf.transform([r['text'] for r in teach_train]), [r['teach'][5] * 100 for r in teach_train])
    pt_tf = ridge_t.predict(tf.transform(tt)); g = np.random.default_rng(1); n = len(tt); d = []
    for _ in range(2000):
        i = g.integers(0, n, n); d.append(spearmanr(st[i], yt[i])[0] - spearmanr(pt_tf[i], yt[i])[0])
    r_rnn, r_tf = spearmanr(st, yt)[0], spearmanr(pt_tf, yt)[0]; top = lambda a: set(np.argsort(-a)[:n // 5])
    RESULTS['E_teacher_fidelity'] = dict(spearman_rnn=float(r_rnn), spearman_tfidf=float(r_tf), diff_ci95=[float(x) for x in np.percentile(d, [2.5, 97.5])],
        mae=float(np.mean(np.abs(st - yt))), bias=float(np.mean(st - yt)), top20_overlap=float(len(top(st) & top(yt)) / (n // 5)), n=int(n))
    print('(E) Fidelidad al profesor en test: Spearman RNN %.3f vs TF-IDF %.3f | dif IC95%% %s | MAE %.1f pts | sesgo %+.1f | coincidencia del top 20%%: %.2f'
          % (r_rnn, r_tf, np.round(np.percentile(d, [2.5, 97.5]), 3), RESULTS['E_teacher_fidelity']['mae'], RESULTS['E_teacher_fidelity']['bias'], RESULTS['E_teacher_fidelity']['top20_overlap']))
else: print('(E) omitido: no hay profesor LLM')

# (F) prueba de sensatez con frases escritas a mano (emocionales vs neutras)
EMO = ["I cannot believe this just happened, it is absolutely insane, I am shaking right now!",
       "She screamed with joy and everyone in the room went completely crazy, it was unbelievable!",
       "This is the worst thing that has ever happened to me, I am furious and I will never forgive him!",
       "Wait, you are telling me he actually quit on live television? No way, that is shocking!",
       "Nobody tells you this about money, and it completely changed my life in just one week.",
       "He looked me in the eye and said the one thing I never expected to hear from my own father."]
NEU = ["Today we will review the quarterly budget table and the assumptions behind each column.",
       "The next step is to open the settings menu and select the second option from the list.",
       "As you can see in the diagram, the system has three components connected by a network.",
       "Let me explain the definition of the term before we move to the next section of the lecture.",
       "Please remember to subscribe to the channel and follow the links in the description below.",
       "In this video I will show the installation steps for the software on a Windows computer."]
se, sn = score_segments(EMO), score_segments(NEU); auc = roc_auc_score([1] * len(EMO) + [0] * len(NEU), se + sn)
RESULTS['F_sanity'] = dict(mean_emotional=float(np.mean(se)), mean_neutral=float(np.mean(sn)), auc=float(auc))
print(f'(F) Frases emocionales: {np.mean(se):.1f} | neutras: {np.mean(sn):.1f} | AUC {auc:.2f}  (v2: 69.8 vs 79.1, al reves)')
if auc < 0.8: print('    AVISO: el modelo aun no separa bien emocional de neutro; revisa N_TEACHER_TRAIN y el profesor LLM.')
print('Ejemplo:', explain(EMO[0]))

## 11. Evaluacion con humanos (conjunto manual de Kevin y Andres)
Ejecuta esta celda cuando tengan `manual_labeled.csv` (columnas: id, text, kevin_score, andres_score). Es la prueba que mas pesa ante el jurado.

In [ ]:
def evaluate_manual(path=None):
    if path is None:
        from google.colab import files; up = files.upload(); path = next(iter(up))
    df = pd.read_csv(path).dropna(subset=['kevin_score', 'andres_score']); h = (df.kevin_score + df.andres_score) / 2
    s = np.array(score_segments(df.text.tolist())); out = dict(n=int(len(df)))
    out['acuerdo_kevin_andres_spearman'] = float(spearmanr(df.kevin_score, df.andres_score)[0])
    out['rnn_vs_humano_spearman'] = float(spearmanr(s, h)[0])
    out['rnn_vs_kevin'] = float(spearmanr(s, df.kevin_score)[0]); out['rnn_vs_andres'] = float(spearmanr(s, df.andres_score)[0])
    out['tfidf_vs_humano_spearman'] = float(spearmanr(Ridge(alpha=3.0).fit(Xtf, Y(main_train, 'adj')).predict(tf.transform(df.text.tolist())), h)[0])
    if USE_TEACHER:
        tch = np.array([(cache.get(ckey(t)) or [np.nan] * 6)[5] * 100 for t in df.text]); ok = np.isfinite(tch)
        if ok.sum() > 10: out['profesor_vs_humano_spearman'] = float(spearmanr(tch[ok], h[ok])[0])
    k = max(len(df) // 5, 1); out['top20_coincidencia_rnn_humano'] = float(len(set(np.argsort(-s)[:k]) & set(np.argsort(-h.values)[:k])) / k)
    g = np.random.default_rng(2); n = len(df); d = [spearmanr(s[i], h.values[i])[0] for i in (g.integers(0, n, n) for _ in range(2000))]
    out['rnn_vs_humano_ic95'] = [float(x) for x in np.percentile(d, [2.5, 97.5])]
    RESULTS['G_manual_humans'] = out; print(json.dumps(out, indent=1)); return out
# evaluate_manual()          # <- descomenta cuando tengan manual_labeled.csv (o llama evaluate_manual('ruta.csv'))

## 12. Exportar artefactos para la app

In [ ]:
os.makedirs('model_out', exist_ok=True)
for k in range(len(MODELS)): shutil.copy(f'model_{k}.pt', f'model_out/model_{k}.pt')
shutil.copy('tokenizer.model', 'model_out/tokenizer.model')
config = dict(version=3, n_models=len(MODELS), vocab=VOCAB, emb=CFG['EMB'], hid=CFG['HID'], layers=CFG['LAYERS'], drop=CFG['DROP'], max_len=MAXL,
              n_go=NGO, n_em=NEM, go_names=GO_NAMES, em_names=EM_NAMES, heads=HEADS, dims=DIMS, position_independent=True,
              uses_teacher=USE_TEACHER, teacher_model=CFG['LLM_MODEL'] if USE_TEACHER else None, score_weights=SW, calibration=CAL,
              window=dict(target=CFG['WIN_TARGET'], min=CFG['WIN_MIN'], max=CFG['WIN_MAX'], hop=CFG['HOP']),
              score='clip(a*(w_tv*tv + w_adj*adj + w_pk*pk)+b, 10, 100)  [metodo linear] | interp por cuantiles [metodo quantile]')
json.dump(config, open('model_out/config.json', 'w'), indent=1)
golden = [dict(text=t, score=s, **{k: float(v[0]) for k, v in predict_texts([t]).items() if k in ('adj', 'pk', 'tv')}) for t, s in zip(EMO + NEU, se + sn)]
json.dump(golden, open('model_out/golden_tests.json', 'w'), indent=1)        # la app debe reproducir estos puntajes (prueba de paridad)
RESULTS['datasets'] = ['behavior-in-the-wild/content-behavior-corpus', 'google-research-datasets/go_emotions', 'dair-ai/emotion']
RESULTS['sizes'] = dict(train_windows=len(main_train), val_windows=len(main_val), test_windows=len(main_test), train_videos=len({r['vid'] for r in main_train}),
                        test_videos=len({r['vid'] for r in main_test}), teacher_labeled=dict(train=len(teach_train), val=len(teach_val), test=len(teach_test)) if USE_TEACHER else 0)
json.dump(RESULTS, open('model_out/metrics.json', 'w'), indent=1, default=str)
if USE_TEACHER:
    pd.DataFrame([dict(text=r['text'], **{d: r['teach'][i] * 10 for i, d in enumerate(DIMS)}, overall=r['teach'][5] * 100) for r in teach_train + teach_val + teach_test]).to_csv('model_out/teacher_labels.csv', index=False)
shutil.copy('manual_template.csv', 'model_out/manual_template.csv')
shutil.make_archive('sparkcut_model_v3', 'zip', 'model_out')
try:
    from google.colab import files; files.download('sparkcut_model_v3.zip')
except Exception: print('Listo: sparkcut_model_v3.zip')